# Exportar LiteFallNet en formato compacto (INT8) para ESP32

Este cuaderno convierte el modelo ya entrenado. No vuelve a entrenar ni cambia el cuaderno anterior.
La cuantización guarda los valores del modelo usando menos bits. LiteRT es el sistema de Google para convertir y ejecutar modelos en dispositivos pequeños.
INT8 significa entero con signo de 8 bits. El ESP32 es el microcontrolador donde se quiere ejecutar el modelo.

## 1. Librerías y archivos

El cuaderno lee el modelo, sus metadatos y los arreglos guardados por el cuaderno de entrenamiento.
Solo usa las 1.000 ventanas originales de entrenamiento para calibrar la cuantización.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.lite.python import schema_py_generated as schema_fb

TAMANO_VENTANA = 150
CANALES = ['AVM', 'GVM']
UMBRAL_CAIDA = 0.5
CAIDA_MAXIMA_PERMITIDA = 0.02

RAIZ_PROYECTO = Path.cwd()
while not (RAIZ_PROYECTO / 'pyproject.toml').is_file() and RAIZ_PROYECTO.parent != RAIZ_PROYECTO:
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if not (RAIZ_PROYECTO / 'pyproject.toml').is_file():
    raise FileNotFoundError('Abre el cuaderno dentro del proyecto.')

RUTA_SALIDA = RAIZ_PROYECTO / 'notebooks' / 'data' / 'modelos' / 'falls' / 'litefallnet_sisfall'
RUTA_MODELO = RUTA_SALIDA / 'litefallnet_sisfall.keras'
RUTA_METADATA = RUTA_SALIDA / 'litefallnet_sisfall_metadata.json'
RUTA_EVALUACION = RUTA_SALIDA / 'litefallnet_sisfall_eval.npz'
RUTA_TFLITE = RUTA_SALIDA / 'litefallnet_sisfall_int8.tflite'
RUTA_HEADER = RUTA_SALIDA / 'litefallnet_sisfall_int8_model.h'
RUTA_CONFIG = RUTA_SALIDA / 'litefallnet_sisfall_int8_config.h'
RUTA_REPORTE = RUTA_SALIDA / 'litefallnet_sisfall_int8_report.json'


## 2. Leer y revisar los datos

AVM resume la aceleración de los tres ejes; GVM, el giro medido por el giroscopio. La entrada ya debe tener estos valores en la escala del entrenamiento; no se vuelven a escalar.
ADL significa actividades de la vida diaria; Fall significa caída.

In [ ]:
def validar_ventanas_y_etiquetas(nombre, ventanas, etiquetas):
    if ventanas.ndim != 3 or ventanas.shape[1:] != (TAMANO_VENTANA, len(CANALES)):
        raise ValueError(f'{nombre}: las señales deben tener forma [N, 150, 2].')
    if etiquetas.ndim != 1:
        raise ValueError(f'{nombre}: las etiquetas deben tener una dimensión.')
    if len(ventanas) == 0 or len(etiquetas) != len(ventanas):
        raise ValueError(f'{nombre}: debe haber una etiqueta por ventana y al menos una ventana.')
    if not np.isin(etiquetas, [0, 1]).all():
        raise ValueError(f'{nombre}: las etiquetas solo pueden ser 0 (ADL) o 1 (caída).')
    if not np.isfinite(ventanas).all():
        raise ValueError(f'{nombre}: las señales contienen valores no finitos.')

In [ ]:
archivos_necesarios = (RUTA_MODELO, RUTA_METADATA, RUTA_EVALUACION)
faltantes = [str(ruta) for ruta in archivos_necesarios if not ruta.is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos de entrenamiento: ' + ', '.join(faltantes))

modelo = tf.keras.models.load_model(RUTA_MODELO, compile=False)
metadata = json.loads(RUTA_METADATA.read_text(encoding='utf-8'))
with np.load(RUTA_EVALUACION, allow_pickle=False) as datos:
    x_calibration = datos['x_calibration'].astype(np.float32)
    y_calibration = datos['y_calibration']
    x_validation = datos['x_validation'].astype(np.float32)
    y_validation = datos['y_validation']

if metadata['canales'] != CANALES or metadata['forma_entrada'] != [TAMANO_VENTANA, len(CANALES)]:
    raise ValueError('Los metadatos no coinciden con la entrada AVM/GVM de 150 muestras.')
validar_ventanas_y_etiquetas('Calibración', x_calibration, y_calibration)
if len(x_calibration) != 1000:
    raise ValueError('La calibración debe contener exactamente 1.000 ventanas de entrenamiento.')
if not np.array_equal(np.bincount(y_calibration.astype(np.int8), minlength=2), [400, 600]):
    raise ValueError('La calibración debe tener 400 ventanas ADL y 600 ventanas de caída.')
validar_ventanas_y_etiquetas('Validación', x_validation, y_validation)
y_calibration = y_calibration.astype(np.int8)
y_validation = y_validation.astype(np.int8)

print(f'Calibración: {len(x_calibration)} ventanas originales; Validación: {len(x_validation)} ventanas.')

## 3. Convertir a enteros de 8 bits (INT8) y guardar el arreglo C

La conversión fija el lote en una ventana de 150 muestras. La entrada y la salida usan enteros con signo de 8 bits (INT8), y el archivo C guarda el modelo como bytes para incluirlo en el programa del microcontrolador.

In [ ]:
def generar_datos_representativos():
    for ventana in x_calibration:
        yield [ventana[np.newaxis, ...].astype(np.float32)]


entrada_fija = tf.keras.Input(batch_shape=(1, TAMANO_VENTANA, len(CANALES)), dtype=tf.float32, name='sensores')
modelo_fijo = tf.keras.Model(entrada_fija, modelo(entrada_fija, training=False))
convertidor = tf.lite.TFLiteConverter.from_keras_model(modelo_fijo)
convertidor.optimizations = [tf.lite.Optimize.DEFAULT]
convertidor.representative_dataset = generar_datos_representativos
convertidor.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
convertidor.inference_input_type = tf.int8
convertidor.inference_output_type = tf.int8
convertidor.allow_custom_ops = False

modelo_int8 = convertidor.convert()


## 4. Revisar el archivo binario del modelo

LiteRT guarda el modelo como FlatBuffer, un archivo binario con los arreglos de números (tensores) y las operaciones. Este prechequeo estático es una revisión del archivo, sin ejecutarlo en la placa; usa la lista de operaciones de Espressif esp-tflite-micro v1.4.1. La lista se basa en las implementaciones de operaciones de [LiteRT Micro](https://github.com/tensorflow/tflite-micro/blob/main/tensorflow/lite/micro/kernels/micro_ops.h) y en [esp-tflite-micro v1.4.1](https://components.espressif.com/components/espressif/esp-tflite-micro/versions/1.4.1).

INT8 es un entero con signo de 8 bits; INT32, uno de 32 bits. La revisión detiene el cuaderno si hay una operación fuera de la lista, una operación personalizada o un tensor de otro tipo. INT32 solo se permite para valores de ajuste (sesgos) o para indicar formas y ejes. Esta revisión no demuestra que el modelo funcione en una placa.

In [ ]:
VERSION_ESP_TFLITE_MICRO = '1.4.1'
OPERADORES_ESP32_LITERT_MICRO = {
    'ADD',
    'CONCATENATION',
    'CONV_2D',
    'DEPTHWISE_CONV_2D',
    'FULLY_CONNECTED',
    'LEAKY_RELU',
    'LOGISTIC',
    'MAX_POOL_2D',
    'MEAN',
    'MUL',
    'PAD',
    'REDUCE_MAX',
    'RESHAPE',
}

modelo_binario = schema_fb.Model.GetRootAsModel(modelo_int8, 0)
if modelo_binario.SubgraphsLength() != 1:
    raise ValueError('El modelo debe tener un solo subgrafo.')
grafo = modelo_binario.Subgraphs(0)
if grafo.InputsLength() != 1 or grafo.OutputsLength() != 1:
    raise ValueError('El modelo debe tener una entrada y una salida.')

TIPOS_TENSOR = {
    valor: nombre
    for nombre, valor in vars(schema_fb.TensorType).items()
    if nombre.isupper() and isinstance(valor, int)
}
TIPOS_PERMITIDOS = {schema_fb.TensorType.INT8, schema_fb.TensorType.INT32}
conteo_tipos = Counter()
tensores_int32 = set()
for indice_tensor in range(grafo.TensorsLength()):
    tensor = grafo.Tensors(indice_tensor)
    tipo = tensor.Type()
    nombre_tipo = TIPOS_TENSOR.get(tipo, f'TIPO_{tipo}')
    conteo_tipos[nombre_tipo] += 1
    if tipo == schema_fb.TensorType.INT32:
        tensores_int32.add(indice_tensor)
    if tipo not in TIPOS_PERMITIDOS:
        raise ValueError(f'El archivo binario contiene un tensor no permitido: {nombre_tipo}.')

CODIGOS_OPERADOR = {
    valor: nombre
    for nombre, valor in vars(schema_fb.BuiltinOperator).items()
    if nombre.isupper() and isinstance(valor, int)
}
nombres_operadores = []
versiones_operadores = {}
OPERADORES_CON_SESGO = {'CONV_2D', 'DEPTHWISE_CONV_2D', 'FULLY_CONNECTED'}
OPERADORES_CON_FORMAS = {'PAD', 'PADV2', 'RESHAPE', 'MEAN', 'REDUCE_MAX'}
usos_int32 = {indice: [] for indice in tensores_int32}
for indice_operador in range(grafo.OperatorsLength()):
    operador = grafo.Operators(indice_operador)
    codigo = modelo_binario.OperatorCodes(operador.OpcodeIndex())
    codigo_builtin = codigo.BuiltinCode()
    if codigo_builtin == schema_fb.BuiltinOperator.CUSTOM:
        nombre_custom = codigo.CustomCode() or b''
        if isinstance(nombre_custom, bytes):
            nombre_custom = nombre_custom.decode('utf-8', errors='replace')
        nombre_operador = f'CUSTOM:{nombre_custom}'
    else:
        nombre_operador = CODIGOS_OPERADOR.get(codigo_builtin, f'BUILTIN_{codigo_builtin}')
    nombres_operadores.append(nombre_operador)
    versiones_operadores.setdefault(nombre_operador, set()).add(int(codigo.Version()))
    for posicion_entrada in range(operador.InputsLength()):
        indice_entrada = operador.Inputs(posicion_entrada)
        if indice_entrada not in tensores_int32:
            continue
        if nombre_operador in OPERADORES_CON_SESGO and posicion_entrada == 2:
            tipo_uso = 'sesgo'
        elif nombre_operador in OPERADORES_CON_FORMAS and posicion_entrada == 1:
            tipo_uso = 'forma_o_eje'
        else:
            raise ValueError(f'El tensor INT32 {indice_entrada} se usa de forma no permitida en {nombre_operador}, entrada {posicion_entrada}.')
        usos_int32[indice_entrada].append({'operador': nombre_operador, 'entrada': posicion_entrada, 'uso': tipo_uso})

for indice_tensor, usos in usos_int32.items():
    if not usos:
        raise ValueError(f'El tensor INT32 {indice_tensor} no tiene un uso permitido.')

operadores_desconocidos = sorted(set(nombres_operadores) - OPERADORES_ESP32_LITERT_MICRO)
if any(nombre.startswith('Flex') or nombre.startswith('CUSTOM:') for nombre in nombres_operadores):
    raise ValueError('El modelo contiene Flex o un operador personalizado.')
if operadores_desconocidos:
    raise ValueError(f'Operadores fuera de la lista LiteRT Micro: {operadores_desconocidos}')

def forma_tensor(tensor):
    return [int(tensor.Shape(indice)) for indice in range(tensor.ShapeLength())]

entrada = grafo.Tensors(grafo.Inputs(0))
salida = grafo.Tensors(grafo.Outputs(0))
forma_entrada = forma_tensor(entrada)
forma_salida = forma_tensor(salida)
if forma_salida != [1, 1]:
    raise ValueError(f'La salida debe tener forma [1, 1], no {forma_salida}.')
if entrada.Type() != schema_fb.TensorType.INT8 or salida.Type() != schema_fb.TensorType.INT8:
    raise ValueError('La entrada y la salida del modelo deben ser INT8.')
if forma_entrada != [1, TAMANO_VENTANA, len(CANALES)]:
    raise ValueError(f'La entrada debe tener lote fijo 1 y forma [1, 150, 2], no {forma_entrada}.')

def leer_parametros_cuantizacion(tensor):
    parametros = tensor.Quantization()
    if parametros is None:
        return {'escalas': [], 'puntos_cero': []}
    escalas = [float(parametros.Scale(i)) for i in range(parametros.ScaleLength())]
    puntos_cero = [int(parametros.ZeroPoint(i)) for i in range(parametros.ZeroPointLength())]
    return {'escalas': escalas, 'puntos_cero': puntos_cero}

cuantizacion_entrada = leer_parametros_cuantizacion(entrada)
cuantizacion_salida = leer_parametros_cuantizacion(salida)
if len(cuantizacion_entrada['escalas']) != 1 or len(cuantizacion_salida['escalas']) != 1:
    raise ValueError('La entrada y la salida deben tener una escala de cuantización.')
if not nombres_operadores:
    raise ValueError('El archivo binario no contiene operaciones.')
conteo_operadores = Counter(nombres_operadores)
inventario_operadores = {
    nombre: {
        'cantidad': int(conteo_operadores[nombre]),
        'versiones_flatbuffer': sorted(versiones_operadores[nombre]),
    }
    for nombre in sorted(conteo_operadores)
}

interprete = tf.lite.Interpreter(model_content=modelo_int8)
interprete.allocate_tensors()
detalle_entrada = interprete.get_input_details()[0]
detalle_salida = interprete.get_output_details()[0]
if detalle_entrada['dtype'] != np.int8 or detalle_salida['dtype'] != np.int8:
    raise ValueError('LiteRT debe informar entrada y salida INT8.')
if tuple(detalle_entrada['shape']) != (1, TAMANO_VENTANA, len(CANALES)):
    raise ValueError('LiteRT no conservó la forma fija de lote 1.')
if tuple(detalle_salida['shape']) != (1, 1):
    raise ValueError('LiteRT no conservó la forma de salida [1, 1].')
escala_entrada, cero_entrada = detalle_entrada['quantization']
escala_salida, cero_salida = detalle_salida['quantization']
if escala_entrada <= 0 or escala_salida <= 0:
    raise ValueError('Las escalas de entrada y salida deben ser mayores que cero.')

RANGO_ENTRADA_MAX = (127 - cero_entrada) * escala_entrada
RANGO_ENTRADA_MIN = (-128 - cero_entrada) * escala_entrada
NORMALIZACION = metadata['normalizacion']


def valor_normalizacion(sufijo):
    """Busca por sufijo: las claves del JSON llevan acentos y no son estables literales."""
    encontrados = [valor for clave, valor in NORMALIZACION.items() if clave.endswith(sufijo)]
    if len(encontrados) != 1:
        raise KeyError(f'Sufijo ambiguo o ausente en metadata[normalizacion]: {sufijo}')
    return encontrados[0]


DIVISORES = [
    float(valor_normalizacion('AVM_divisor')),
    float(valor_normalizacion('GVM_divisor')),
]
UNIDADES = ['g', 'grados_por_segundo']
TOPE_CANAL = [
    float(valor_normalizacion('AVM_límite_por_eje_g')) * np.sqrt(3.0),
    float(valor_normalizacion('GVM_límite_por_eje_grados_segundo')) * np.sqrt(3.0),
]

CUBRE_RANGO_FISICO = RANGO_ENTRADA_MAX >= 1.0 - 1e-9


def medir_saturacion(ventanas):
    """Porcentaje de muestras que exceden el rango representable en int8."""
    cuantizada = np.rint(ventanas.astype(np.float32) / escala_entrada + cero_entrada)
    excede = (cuantizada > 127) | (cuantizada < -128)
    por_ventana = excede.reshape(len(ventanas), -1).any(axis=1)
    return {
        'ventanas': int(len(ventanas)),
        'muestras_excedidas': int(excede.sum()),
        'muestras_totales': int(excede.size),
        'porcentaje_muestras': float(100.0 * excede.sum() / max(excede.size, 1)),
        'ventanas_afectadas': int(por_ventana.sum()),
        'porcentaje_ventanas': float(100.0 * por_ventana.sum() / max(len(ventanas), 1)),
    }


mediciones_saturacion = {
    'calibracion': medir_saturacion(x_calibration),
    'validacion': medir_saturacion(x_validation),
}
for nombre, medicion in mediciones_saturacion.items():
    if medicion['muestras_excedidas']:
        raise ValueError(
            f'{nombre}: {medicion["muestras_excedidas"]} muestras '
            f'({medicion["porcentaje_muestras"]:.4f} %) superan el techo de entrada '
            f'{RANGO_ENTRADA_MAX:.6f}.'
        )
print('Techo de entrada (normalizado): {:.6f} | AVM {:.2f} {} | GVM {:.1f} {}'.format(
    RANGO_ENTRADA_MAX,
    RANGO_ENTRADA_MAX * DIVISORES[0], UNIDADES[0],
    RANGO_ENTRADA_MAX * DIVISORES[1], UNIDADES[1],
))
print(f'Saturacion calibracion/validacion: {mediciones_saturacion["calibracion"]["porcentaje_muestras"]:.4f} % / {mediciones_saturacion["validacion"]["porcentaje_muestras"]:.4f} %')

UMBRAL_SALIDA_INT8 = int(np.clip(np.rint(UMBRAL_CAIDA / escala_salida + cero_salida), -128, 127))
UMBRAL_RECONSTRUIDO = (UMBRAL_SALIDA_INT8 - cero_salida) * escala_salida

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_TFLITE.write_bytes(modelo_int8)
lineas_bytes = [
    '    ' + ', '.join(f'0x{byte:02x}' for byte in modelo_int8[inicio:inicio + 12]) + ','
    for inicio in range(0, len(modelo_int8), 12)
]
nombre_arreglo = 'litefallnet_sisfall_int8_model'
contenido_header = '\n'.join([
    '#ifndef LITEFALLNET_SISFALL_INT8_MODEL_H',
    '#define LITEFALLNET_SISFALL_INT8_MODEL_H',
    '',
    '/* Generado por notebooks/04_Cuantizacion_LiteFallNet_LiteRT_ESP32.ipynb. No editar a mano. */',
    '',
    '/* alignas es C11/C++; en C puro <stdint.h> no la define. */',
    '#include <stdalign.h>',
    '',
    'alignas(16) static const unsigned char ' + nombre_arreglo + '[] = {',
    *lineas_bytes,
    '};',
    'static const unsigned int ' + nombre_arreglo + '_len =',
    '    (unsigned int)sizeof(' + nombre_arreglo + ');',
    '',
    '#endif',
    '',
])
RUTA_HEADER.write_text(contenido_header, encoding='utf-8')

FRECUENCIA_HZ = int(metadata['ventana']['frecuencia_hz'])
PASO_MUESTRAS = int(metadata['ventana']['paso_muestras'])
NUM_CANALES = len(CANALES)


def numero(valor):
    """Representa un float en C sin perder digitos ni notacion exponencial ambigua.

    El sufijo ``f`` es obligatorio: sin el, cada constante ocupa 8 bytes de flash.
    """
    return f"{valor:.17g}f"


contenido_config = f"""#ifndef LITEFALLNET_SISFALL_INT8_CONFIG_H
#define LITEFALLNET_SISFALL_INT8_CONFIG_H

/* Generado por notebooks/04_Cuantizacion_LiteFallNet_LiteRT_ESP32.ipynb. No editar a mano. */

#define LITEFALLNET_FRECUENCIA_HZ {FRECUENCIA_HZ}
#define LITEFALLNET_TAMANO_VENTANA {TAMANO_VENTANA}
#define LITEFALLNET_PASO_MUESTRAS {PASO_MUESTRAS}
#define LITEFALLNET_NUM_CANALES {NUM_CANALES}
#define LITEFALLNET_FORMA_ENTRADA {{1, {TAMANO_VENTANA}, {NUM_CANALES}}}

/* Canal 0 = AVM (g), canal 1 = GVM (grados/s). Dividir antes de cuantizar. */
#define LITEFALLNET_CANAL_AVM 0
#define LITEFALLNET_CANAL_GVM 1
#define LITEFALLNET_AVM_DIVISORF {numero(DIVISORES[0])}
#define LITEFALLNET_GVM_DIVISORF {numero(DIVISORES[1])}
#define LITEFALLNET_AVM_TOPE_FISICO {numero(TOPE_CANAL[0])}
#define LITEFALLNET_GVM_TOPE_FISICO {numero(TOPE_CANAL[1])}

/* Cuantizacion de entrada: q = clip(rint(x / ESCALA + CERO), -128, 127). */
#define LITEFALLNET_ENTRADA_ESCALAF {numero(escala_entrada)}
#define LITEFALLNET_ENTRADA_CERO {cero_entrada}
#define LITEFALLNET_TECHO_ENTRADA_REALF {numero(RANGO_ENTRADA_MAX)}

/* Cuantizacion de salida: p = (q - CERO) * ESCALA, con p en [0, 1]. */
#define LITEFALLNET_SALIDA_ESCALAF {numero(escala_salida)}
#define LITEFALLNET_SALIDA_CERO {cero_salida}

#define LITEFALLNET_UMBRAL_CAIDA {numero(UMBRAL_CAIDA)}
#define LITEFALLNET_UMBRAL_SALIDA_INT8 {UMBRAL_SALIDA_INT8}
#define LITEFALLNET_UMBRAL_RECONSTRUIDO {numero(UMBRAL_RECONSTRUIDO)}

#endif
"""
RUTA_CONFIG.write_text(contenido_config, encoding='utf-8')

print(f'Modelo INT8: {RUTA_TFLITE} ({len(modelo_int8):,} bytes)')
print(f'Arreglo C: {RUTA_HEADER}')
print(f'Config C: {RUTA_CONFIG}')
print(f'Tipos de tensor: {dict(sorted(conteo_tipos.items()))}')
print(f'Operadores: {inventario_operadores}')
print(f'Entrada INT8: {forma_entrada}; salida INT8: {forma_salida}')


## 5. Comparar en validación

Primero se compara el modelo original con INT8 usando solo validación. Exactitud es el porcentaje de aciertos; el recall de caídas indica cuántas caídas detecta el modelo, y F1 resume el equilibrio entre precisión y detección. Si alguna métrica baja más de 2 puntos porcentuales, el modelo no está listo. Haría falta un ajuste especial durante el entrenamiento, llamado QAT (entrenamiento adaptado a la cuantización); este cuaderno no lo ejecuta.

In [ ]:
def calcular_metricas(y_real, probabilidades):
    y_real = np.asarray(y_real, dtype=np.int8).ravel()
    predicciones = (np.asarray(probabilidades).ravel() >= UMBRAL_CAIDA).astype(np.int8)
    verdaderos_negativos = int(np.sum((y_real == 0) & (predicciones == 0)))
    falsos_positivos = int(np.sum((y_real == 0) & (predicciones == 1)))
    falsos_negativos = int(np.sum((y_real == 1) & (predicciones == 0)))
    verdaderos_positivos = int(np.sum((y_real == 1) & (predicciones == 1)))
    precision = verdaderos_positivos / (verdaderos_positivos + falsos_positivos) if verdaderos_positivos + falsos_positivos else 0.0
    recall = verdaderos_positivos / (verdaderos_positivos + falsos_negativos) if verdaderos_positivos + falsos_negativos else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    total = len(y_real)
    return {
        'accuracy': float((verdaderos_negativos + verdaderos_positivos) / total),
        'precision': float(precision),
        'recall_caidas': float(recall),
        'specificity': float(verdaderos_negativos / (verdaderos_negativos + falsos_positivos)) if verdaderos_negativos + falsos_positivos else 0.0,
        'f1': float(f1),
    }

def inferir_int8(ventanas):
    probabilidades = np.empty(len(ventanas), dtype=np.float32)
    for indice, ventana in enumerate(ventanas):
        entrada_float = np.asarray(ventana, dtype=np.float32)[np.newaxis, ...]
        entrada_int8 = np.rint(entrada_float / escala_entrada + cero_entrada)
        entrada_int8 = np.clip(entrada_int8, -128, 127).astype(np.int8)
        interprete.set_tensor(detalle_entrada['index'], entrada_int8)
        interprete.invoke()
        salida_int8 = interprete.get_tensor(detalle_salida['index']).astype(np.float32)
        probabilidades[indice] = ((salida_int8.ravel()[0] - cero_salida) * escala_salida)
    return probabilidades

probabilidades_keras_validacion = modelo.predict(x_validation, batch_size=32, verbose=0).ravel()
probabilidades_int8_validacion = inferir_int8(x_validation)
metricas_keras_validacion = calcular_metricas(y_validation, probabilidades_keras_validacion)
metricas_int8_validacion = calcular_metricas(y_validation, probabilidades_int8_validacion)
metricas_comparadas = ('accuracy', 'recall_caidas', 'f1')
perdidas_validacion = {
    nombre: metricas_keras_validacion[nombre] - metricas_int8_validacion[nombre]
    for nombre in metricas_comparadas
}
metricas_fuera_del_limite = [
    nombre for nombre, perdida in perdidas_validacion.items()
    if perdida > CAIDA_MAXIMA_PERMITIDA
]
candidato_listo = not metricas_fuera_del_limite
print(f'Validación Keras: {metricas_keras_validacion}')
print(f'Validación INT8: {metricas_int8_validacion}')
print(f'Caídas en métricas: {perdidas_validacion}')
print('Candidato listo para la siguiente evaluación.' if candidato_listo else f'Candidato no listo: revisar {metricas_fuera_del_limite}. Hace falta un ajuste especial durante el entrenamiento para adaptarlo a INT8.')

## 6. Evaluar prueba y UP-Fall solo si se aprueba la validación

Si el candidato no está listo, estos arreglos no se cargan ni se evalúan. En ese caso el informe deja ambas evaluaciones vacías.

In [ ]:
evaluaciones_finales = {'sisfall_prueba': None, 'upfall': None}
metricas_fuera_del_limite_externo = []
perdidas_externas = {}
candidato_despliegue = False
if candidato_listo:
    with np.load(RUTA_EVALUACION, allow_pickle=False) as datos:
        grupos_finales = {
            'sisfall_prueba': (datos['x_test'].astype(np.float32), datos['y_test']),
            'upfall': (datos['x_upfall'].astype(np.float32), datos['y_upfall']),
        }
    for nombre, (x_grupo, y_grupo) in grupos_finales.items():
        validar_ventanas_y_etiquetas(nombre, x_grupo, y_grupo)
    grupos_finales = {
        nombre: (x_grupo, y_grupo.astype(np.int8))
        for nombre, (x_grupo, y_grupo) in grupos_finales.items()
    }
    for nombre, (x_grupo, y_grupo) in grupos_finales.items():
        probabilidades_keras = modelo.predict(x_grupo, batch_size=32, verbose=0).ravel()
        probabilidades_int8 = inferir_int8(x_grupo)
        metricas_keras = calcular_metricas(y_grupo, probabilidades_keras)
        metricas_int8 = calcular_metricas(y_grupo, probabilidades_int8)
        evaluaciones_finales[nombre] = {
            'keras': metricas_keras,
            'int8': metricas_int8,
            'diferencia_int8_menos_keras': {
                metrica: float(metricas_int8[metrica] - metricas_keras[metrica])
                for metrica in metricas_keras
            },
            'saturacion': medir_saturacion(x_grupo),
        }
        print(f'{nombre}: Keras={metricas_keras}; INT8={metricas_int8}')

    # La evaluacion externa es condicion de despliegue, no un chequeo posterior: un modelo
    # que solo aprueba en su propio dataset no se puede desplegar.
    metricas_con_limite_externo = ('precision', 'recall_caidas')
    for nombre in ('sisfall_prueba', 'upfall'):
        evaluacion = evaluaciones_finales[nombre]
        perdidas_externas[nombre] = {
            metrica: float(evaluacion['keras'][metrica] - evaluacion['int8'][metrica])
            for metrica in metricas_con_limite_externo
        }
        for metrica, perdida in perdidas_externas[nombre].items():
            if perdida > CAIDA_MAXIMA_PERMITIDA:
                metricas_fuera_del_limite_externo.append(f'{nombre}.{metrica}')
    candidato_despliegue = not metricas_fuera_del_limite_externo
    print(f'Perdidas externas: {perdidas_externas}')
    print('Candidato apto para despliegue.' if candidato_despliegue else f'No apto para despliegue: {metricas_fuera_del_limite_externo}')
else:
    print('No se cargan ni evaluan SisFall de prueba o UP-Fall: la validacion INT8 no paso.')


## 7. Guardar el informe

El informe guarda el resultado de validación, las evaluaciones permitidas, el tamaño del modelo, sus operadores y las escalas de entrada y salida.

En el ESP32 elegido (un microcontrolador) se debe compilar el firmware, es decir, el programa que corre en la placa, y comprobar el soporte real de las operaciones. También se debe medir la memoria fija para guardar los tensores (arreglos de números) y el tiempo de respuesta. En el programa de la placa, el espacio reservado para ellos se llama arena. `esp-tflite-micro` integra ESP-NN, una biblioteca de optimización de Espressif; `-DESP_NN` activa esas optimizaciones al compilar. Este prechequeo no demuestra que el modelo funcione en la placa ni crea archivos de firmware.

In [ ]:
preprocesamiento = {
    'canales': metadata['canales'],
    'forma_ventana': metadata['forma_entrada'],
    'frecuencia_hz': metadata['ventana']['frecuencia_hz'],
    'paso_muestras': metadata['ventana']['paso_muestras'],
    'normalizacion': metadata['normalizacion'],
    'etiquetas': metadata['etiquetas'],
    'umbral_caida': UMBRAL_CAIDA,
    'umbral_salida_int8': UMBRAL_SALIDA_INT8,
    'umbral_reconstruido': UMBRAL_RECONSTRUIDO,
    'entrada_tflite': {'forma': forma_entrada, 'tipo': 'int8'},
    'salida_tflite': {'forma': forma_salida, 'tipo': 'int8'},
}
if candidato_despliegue:
    estado = 'apto_para_despliegue'
    motivo = None
elif not candidato_listo:
    estado = 'no_listo_para_despliegue'
    motivo = (
        'La cuantizacion INT8 pierde mas de 2 puntos porcentuales en validacion. '
        'Hace falta un ajuste especial durante el entrenamiento para adaptarlo a INT8.'
    )
else:
    estado = 'no_listo_para_despliegue'
    motivo = (
        'La cuantizacion INT8 pierde mas de 2 puntos porcentuales fuera del dataset de '
        f'entrenamiento: {metricas_fuera_del_limite_externo}.'
    )
informe = {
    'estado_candidato': estado,
    'motivo_no_listo': motivo,
    'gate': {
        'validacion_ks': {
            'metricas': list(metricas_comparadas),
            'perdidas': {nombre: float(valor) for nombre, valor in perdidas_validacion.items()},
            'fuera_del_limite': metricas_fuera_del_limite,
            'limite_permitido': CAIDA_MAXIMA_PERMITIDA,
        },
        'evaluacion_externa_obligatoria': True,
        'perdidas_externas': perdidas_externas,
        'fuera_del_limite_externo': metricas_fuera_del_limite_externo,
    },
    'conversion': {
        'optimizaciones': ['DEFAULT'],
        'operadores_permitidos_por_convertidor': ['TFLITE_BUILTINS_INT8'],
        'tipo_entrada': 'int8',
        'tipo_salida': 'int8',
        'forma_entrada': forma_entrada,
        'parametros_cuantizacion_entrada': cuantizacion_entrada,
        'parametros_cuantizacion_salida': cuantizacion_salida,
        'representativo': {'ventanas': int(len(x_calibration))},
        'tipos_tensor': dict(sorted(conteo_tipos.items())),
        'prechequeo_estatico': True,
        'componente_esp_tflite_micro': 'espressif/esp-tflite-micro',
        'version_esp_tflite_micro': VERSION_ESP_TFLITE_MICRO,
        'nombres_operadores_builtin': sorted(inventario_operadores),
        'operadores': inventario_operadores,
        'lista_permitida_esp32_litert_micro': sorted(OPERADORES_ESP32_LITERT_MICRO),
        'usos_int32_permitidos': {str(indice): usos for indice, usos in usos_int32.items()},
    },
    'saturacion_entrada': {
        'rango_representable_normalizado': [RANGO_ENTRADA_MIN, RANGO_ENTRADA_MAX],
        'cubre_rango_fisico_normalizado': bool(CUBRE_RANGO_FISICO),
        'techo_fisico_por_canal': {
            CANALES[indice]: {
                'unidad': UNIDADES[indice],
                'valor': float(RANGO_ENTRADA_MAX * TOPE_CANAL[indice]),
                'sensor': float(TOPE_CANAL[indice]),
            }
            for indice in range(len(CANALES))
        },
        'mediciones': mediciones_saturacion,
    },
    'validacion': {
        'keras': metricas_keras_validacion,
        'int8': metricas_int8_validacion,
        'diferencia_int8_menos_keras': {nombre: float(-valor) for nombre, valor in perdidas_validacion.items()},
        'metricas_con_limite_de_2_puntos': list(metricas_comparadas),
        'metricas_fuera_del_limite': metricas_fuera_del_limite,
        'limite_permitido': CAIDA_MAXIMA_PERMITIDA,
    },
    'sisfall_prueba': evaluaciones_finales['sisfall_prueba'],
    'upfall': evaluaciones_finales['upfall'],
    'preprocesamiento': preprocesamiento,
    'archivos': {
        'modelo_tflite': RUTA_TFLITE.name,
        'arreglo_c': RUTA_HEADER.name,
        'config_c': RUTA_CONFIG.name,
        'informe': RUTA_REPORTE.name,
        'tamano_modelo_bytes': len(modelo_int8),
        'tamano_modelo_kib': round(len(modelo_int8) / 1024, 2),
        'tamano_config_bytes': RUTA_CONFIG.stat().st_size,
    },
    'nota_placa': (
        'Este prechequeo estatico no demuestra el soporte real en una placa. '
        'Compila firmware para el ESP32 elegido y mide el soporte de operaciones, '
        'la memoria fija para guardar tensores (arena) y la latencia. '
        'esp-tflite-micro integra ESP-NN y -DESP_NN activa sus optimizaciones. '
        'El arreglo C del modelo se declara alignas(16); los kernels optimizados de '
        'esp-nn asumen esa alineacion.'
    ),
}
RUTA_REPORTE.write_text(json.dumps(informe, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Informe: {RUTA_REPORTE}')
print('Estado: {}; tamano: {} KiB'.format(informe['estado_candidato'], informe['archivos']['tamano_modelo_kib']))
